# <center> 机器学习实验课: 多层感知器 </center>
#### <center>  助教：王鑫 </center>


# 1. Artificial Neural Networks (人工神经网络)

<p style="text-align: justify;">人工神经网络是受人脑启发的数学模型，特别是学习、处理和执行任务的能力。人工神经网络是协助解决复杂问题的强大工具，主要与组合优化和机器学习领域有关。在这种情况下，人工神经网络具有最多样化的应用，因为这种模型可以适应所呈现的情况，确保在没有任何人类干扰的情况下逐步提高性能。我们可以说，人工神经网络是有力的方法，可以给计算机一个新的可能性，也就是说，机器不会拘泥于预先编程的规则，并开辟各种选择，从自己的错误中学习。 </p>


# 2. 如何实现多层感知器

## 2.1. Some Python Libraries 

<p style="text-align: justify;">首先，我们可以使用一些库来帮助我们处理数据集，如 "numpy"、"matplotlib"、"seaborn "和 "scikit-learn"。在本教程中，我们考虑在没有使用任何像Keras或类似的框架下来实现一个多层感知器。这里的目标是尽可能的简单，因此，为了帮助你完成这项任务，你可以使用numpy来处理数组操作! </p>

In [ ]:
import random
import seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

seaborn.set(style='whitegrid'); seaborn.set_context('talk')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from sklearn.datasets import load_iris
iris_data = load_iris()

## 2.2. An analysis about the Iris Flower Dataset


In [ ]:
print(iris_data['DESCR'])

In [ ]:
n_samples, n_features = iris_data.data.shape

plt.subplot(1, 2, 1)
scatter_plot = plt.scatter(iris_data.data[:,0], iris_data.data[:,1], alpha=0.5, 
                           c=iris_data.target) 
plt.colorbar(ticks=([0, 1, 2]))
plt.title('Sepal Sample')

plt.subplot(1, 2, 2)
scatter_plot_2 = plt.scatter(iris_data.data[:,2], iris_data.data[:,3], alpha=0.5, 
                           c=iris_data.target)
plt.colorbar(ticks=([0, 1, 2]))
plt.title('Petal Sample')

In [ ]:
import pandas
from pandas.plotting import scatter_matrix


dataset = pandas.read_csv('../input/iris/Iris.csv')
dataset

In [ ]:
scatter_matrix(dataset, alpha=0.5, figsize=(20, 20))
plt.show()

In [ ]:
dataset.hist(alpha=0.5, figsize=(20, 20), color='red')
plt.show()

In [ ]:
dataset.plot(subplots=True, figsize=(10, 10), sharex=False, sharey=False)
plt.show()

# 3. Manually separating our dataset

It is here that we will select our samples to train and test the algorithms: **80% Training Samples and 20% Test**
<div class="container-fluid">
  <div class="row">
      <div class="col-md-2" align='center'>
      </div>
      <div class='col-md-8' align='center'>
      </div>
      <div class="col-md-2" align='center'></div>
  </div>
</div>

In [ ]:
random.seed(123)

def separate_data():
    A = iris_dataset[0:40]
    tA = iris_dataset[40:50]
    B = iris_dataset[50:90]
    tB = iris_dataset[90:100]
    C = iris_dataset[100:140]
    tC = iris_dataset[140:150]
    train = np.concatenate((A,B,C))
    test =  np.concatenate((tA,tB,tC))
    return train,test

train_porcent = 80 # Porcent Training 
test_porcent = 20 # Porcent Test

iris_dataset = np.column_stack((iris_data.data,iris_data.target.T)) #Join X and Y
iris_dataset = list(iris_dataset)
random.shuffle(iris_dataset)

Filetrain, Filetest = separate_data()

train_X = np.array([i[:4] for i in Filetrain])
train_y = np.array([i[4] for i in Filetrain])
test_X = np.array([i[:4] for i in Filetest])
test_y = np.array([i[4] for i in Filetest])

## 3.1. Plot our training Samples

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm


plt.subplot(1, 2, 1)
plt.scatter(train_X[:,0],train_X[:,1],c=train_y,cmap=cm.viridis)
plt.xlabel(iris_data.feature_names[0])
plt.ylabel(iris_data.feature_names[1])

plt.subplot(1, 2, 2)
plt.scatter(train_X[:,2],train_X[:,3],c=train_y,cmap=cm.viridis)
plt.xlabel(iris_data.feature_names[2])
plt.ylabel(iris_data.feature_names[3])

## 3.2. Plot our test Samples

In [ ]:
plt.subplot(1, 2, 1)
plt.scatter(test_X[:,0],test_X[:,1],c=test_y,cmap=cm.viridis)
plt.xlabel(iris_data.feature_names[0])
plt.ylabel(iris_data.feature_names[1]) 

plt.subplot(1, 2, 2)
plt.scatter(test_X[:,2],test_X[:,3],c=test_y,cmap=cm.viridis)
plt.xlabel(iris_data.feature_names[2])
plt.ylabel(iris_data.feature_names[3])



# 4. Multilayer Perceptron

<p style="text-align: justify;">人工神经网络（ANNs）或连接主义系统是受构成动物大脑的生物神经网络启发的计算系统。这类系统通过考虑实例来学习（逐步提高性能）完成任务，通常不需要特定的任务编程。例如，在图像识别中，"它们可以通过分析被手动标记为 "猫 "或 "无猫 "的示例图像来学习识别含有猫的图像，并使用分析结果来识别其他图像中的猫"</p>

<p style="text-align: justify;">它们在难以用基于规则的编程的传统计算机算法表达的应用中得到了最多的使用。人工神经网络的基础是被称为人工神经元的连接单元的集合，（类似于生物大脑中的轴突）。神经元之间的每个连接（突触）可以向另一个神经元传输信号。接收（突触后）的神经元可以处理信号，然后向与其连接的下游神经元发出信号。</p>

<p style="text-align: justify;"> More information here: [Artificial Neural Network](https://en.wikipedia.org/wiki/Artificial_neural_network)</p>

<img src="https://miro.medium.com/max/1072/1*DOkHU_dgXMCybA6WWXrp4g.gif"/>

<p style="text-align: justify;">多层感知器网络的特点是在你的结构中存在许多中间层（隐藏），位于输入层和输出层之间。有了这一点，这种网络的优点是能够对两个以上的不同类别进行分类，它还能解决非线性可分离的问题。</p>

<center> <img src="https://cdn-images-1.medium.com/max/800/0*eaw1POHESc--l5yR.png"/> <\center>


## 4.1. How does Multilayer Perceptron work? 

<p style="text-align: justify;"> We can summarize the operation of the perceptron as follows it:</p>

  - **Step 1**: 用小的随机值初始化权重和偏置
  - **Step 2**: 将输入层的所有数值传播到输出层（正向传播）
  - **Step 3**: 更新层间的权重和偏差（反向传播）
  - **Step 4**: 重复2,3直到最终损失收敛
  
### Step 1: Forward propagation Algorithm
<img src="https://sebastianraschka.com/images/faq/visual-backpropagation/forward-propagation.png">

In order to proceed we need to improve the notation we have been using. That for, for each layer $1\geq l\geq L$, the activations and outputs are calculated as:

$$
\text{L}^l_j = {\sum_i w^l_{ji} x^l_i\, = w^l_{j,0} x^l_0 + w^l_{j,1} x^l_1 + w^l_{j,2} x^l_2 + ... + w^l_{j,n}} x^l_n,
$$
$$Y^l_j = g^l(\text{L}^l_j)\,,$$

$$\{y_{i},\,x_{i1},\ldots ,x_{ip}\}_{i=1}^{n}$$

where:

* $y^l_j$ is the $j-$th output of layer $l$,
* $x^l_i$ is the $i$-th input to layer $l$,
* $w^l_{ji}$ is the weight of the $j$-th neuron connected to input $i$,
* $\text{L}^l_{j}$ is called net activation, and
* $g^l(\cdot)$ is the activation function of layer $l$.

### Step 2. Activation Functions
<img src="https://miro.medium.com/max/1192/1*4ZEDRpFuCIpUjNgjDdT2Lg.png">

### Sigmoid Function:

In [ ]:
x = 0 
ativation = {(lambda x: 1/(1 + np.exp(-x)))}
deriv = {(lambda x: x*(1-x))}

 ### Hyperbolic Tangent Function:

In [ ]:
activation_tang = {(lambda x: np.tanh(x))}
deriv_tang = {(lambda x: 1-x**2)}
  

### ReLU Function:

In [ ]:
activation_ReLU = {(lambda x: x*(x > 0))}
deriv_ReLU = {(lambda x: 1 * (x>0))}

### Step 3: Calculation our Erro function 
<img src="https://miro.medium.com/max/920/1*jYQYuHpHdkZqNFQKJSuDTw.png">
It is used to measure performance locality associated with the results produced by the neurons in output layer and the expected result.
$$
E(k) = 
\frac{1}{2} \sum_{k=1}^{K}({{d_j(k)}} - {y_j}{(k)})^2.
$$

### Step 4. Backpropagation Algorithm
<img src="https://sebastianraschka.com/images/faq/visual-backpropagation/backpropagation.png">
### In Output Layer,  $L = 2:$
   - **Step 2**: Calculate error in output layer: $\delta^{(L2)} = -({d_j}^{(L2)} - {y_j}^{(L2)})\cdot
   g'({S_j}^{(L2)})$
   
      `
      ERROR_output = self.OUTPUT - self.OUTPUT_L2
      DELTA_output = ((-1)*(ERROR_output) * self.deriv(self.OUTPUT_L2))
      `
      

   - **Step 2**: Update all weight between hidden and output layer: $W^{(L2)} = W^{(L2)} -\gamma \cdot(\delta^{(L2)}  - {S_j}^{(L1)})$
   
         for i in range(self.hiddenLayer):`
           ` for j in range(self.OutputLayer):`
               ` self.WEIGHT_output[i][j] -= (self.learningRate * (DELTA_output[j] * self.output_l1[i]))`
               ` self.BIAS_output[j] -= (self.learningRate * DELTA_output[j])`
               
   - **Step 3**: Update bias value in output layer: $bias^{(L2)} = bias^{(L2)} - \gamma \cdot \delta^{(L2)}$
   
### In Input Layer , $L = 1$:
   - **Step 4**: Calculate error in hidden layer: $\delta^{(L1)} = W^{(L2)} \cdot \delta^{(L2)} \cdot g'({S_j}^{(L1)})$
     
   `delta_hidden = np.matmul(self.WEIGHT_output, DELTA_output) * self.deriv(self._l1)`
   - **Step 5**: Update all weight between hidden and output layer: $W^{(L1)} = W^{(L1)} -\gamma \cdot(\delta^{(L1)}  - {X_i})$
         `for i in range(self.OutputLayer):`
           `for j in range(self.hiddenLayer):`
               `self.WEIGHT_hidden[i][j] -= (self.learningRate * (DELTA_hidden[j] * INPUT[i]))`
               `self.BIAS_hidden[j] -= (self.learningRate * DELTA_hidden[j])`
   - **Step 6**: Update bias value in output layer: $bias^{(L1)} = bias^{(L1)} - \gamma \cdot \delta^{(L1)}$

<img src="https://thumbs.gfycat.com/FickleHorribleBlackfootedferret-small.gif">

# 5. 使用Python实现多层感知器


In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
import random

class MultiLayerPerceptron(BaseEstimator, ClassifierMixin): 
    def __init__(self, params=None):     
        if (params == None):
            self.inputLayer = 4                        # Input Layer
            self.hiddenLayer = 5                       # Hidden Layer
            self.outputLayer = 3                       # Outpuy Layer
            self.learningRate = 0.005                  # Learning rate
            self.max_epochs = 600                      # Epochs
            self.iasHiddenValue = -1                   # Bias HiddenLayer
            self.BiasOutputValue = -1                  # Bias OutputLayer
            self.activation = self.ativacao['sigmoid'] # Activation function
            self.deriv = self.derivada['sigmoid']
        else:
            self.inputLayer = params['InputLayer']
            self.hiddenLayer = params['HiddenLayer']
            self.OutputLayer = params['OutputLayer']
            self.learningRate = params['LearningRate']
            self.max_epochs = params['Epocas']
            self.BiasHiddenValue = params['BiasHiddenValue']
            self.BiasOutputValue = params['BiasOutputValue']
            self.activation = self.ativacao[params['ActivationFunction']]
            self.deriv = self.derivada[params['ActivationFunction']]
        
        'Starting Bias and Weights'
        self.WEIGHT_hidden = self.starting_weights(self.hiddenLayer, self.inputLayer)
        self.WEIGHT_output = self.starting_weights(self.OutputLayer, self.hiddenLayer)
        self.BIAS_hidden = np.array([self.BiasHiddenValue for i in range(self.hiddenLayer)])
        self.BIAS_output = np.array([self.BiasOutputValue for i in range(self.OutputLayer)])
        self.classes_number = 3 
        
    pass
    
    def starting_weights(self, x, y):
        return [[2  * random.random() - 1 for i in range(x)] for j in range(y)]

    ativacao = {
         'sigmoid': (lambda x: 1/(1 + np.exp(-x))),
            'tanh': (lambda x: np.tanh(x)),
            'Relu': (lambda x: x*(x > 0)),
               }
    derivada = {
         'sigmoid': (lambda x: x*(1-x)),
            'tanh': (lambda x: 1-x**2),
            'Relu': (lambda x: 1 * (x>0))
               }
 
    def Backpropagation_Algorithm(self, x):
        DELTA_output = []
        'Stage 1 - Error: OutputLayer'
        ERROR_output = self.output - self.OUTPUT_L2
        DELTA_output = ((-1)*(ERROR_output) * self.deriv(self.OUTPUT_L2))
        
        arrayStore = []
        'Stage 2 - Update weights OutputLayer and HiddenLayer'
        for i in range(self.hiddenLayer):
            for j in range(self.OutputLayer):
                self.WEIGHT_output[i][j] -= (self.learningRate * (DELTA_output[j] * self.OUTPUT_L1[i]))
                self.BIAS_output[j] -= (self.learningRate * DELTA_output[j])
      
        'Stage 3 - Error: HiddenLayer'
        delta_hidden = np.matmul(self.WEIGHT_output, DELTA_output)* self.deriv(self.OUTPUT_L1)
 
        'Stage 4 - Update weights HiddenLayer and InputLayer(x)'
        for i in range(self.OutputLayer):
            for j in range(self.hiddenLayer):
                self.WEIGHT_hidden[i][j] -= (self.learningRate * (delta_hidden[j] * x[i]))
                self.BIAS_hidden[j] -= (self.learningRate * delta_hidden[j])
                
    def show_err_graphic(self,v_erro,v_epoca):
        plt.figure(figsize=(9,4))
        plt.plot(v_epoca, v_erro, "m-",color="b", marker=11)
        plt.xlabel("Number of Epochs")
        plt.ylabel("Squared error (MSE) ");
        plt.title("Error Minimization")
        plt.show()

    def predict(self, X, y):
        'Returns the predictions for every element of X'
        my_predictions = []
        'Forward Propagation'
        forward = np.matmul(X,self.WEIGHT_hidden) + self.BIAS_hidden
        forward = np.matmul(forward, self.WEIGHT_output) + self.BIAS_output
                                 
        for i in forward:
            my_predictions.append(max(enumerate(i), key=lambda x:x[1])[0])
            
        array_score = []
        for i in range(len(my_predictions)):
            if my_predictions[i] == 0: 
                array_score.append([i, 'Iris-setosa', my_predictions[i], y[i]])
            elif my_predictions[i] == 1:
                 array_score.append([i, 'Iris-versicolour', my_predictions[i], y[i]])
            elif my_predictions[i] == 2:
                 array_score.append([i, 'Iris-virginica', my_predictions[i], y[i]])
                    
        dataframe = pd.DataFrame(array_score, columns=['_id', 'class', 'output', 'hoped_output'])
        return my_predictions, dataframe

    def fit(self, X, y):  
        count_epoch = 1
        total_error = 0
        n = len(X); 
        epoch_array = []
        error_array = []
        W0 = []
        W1 = []
        while(count_epoch <= self.max_epochs):
            for idx,inputs in enumerate(X): 
                self.output = np.zeros(self.classes_number)
                'Stage 1 - (Forward Propagation)'
                self.OUTPUT_L1 = self.activation((np.dot(inputs, self.WEIGHT_hidden) + self.BIAS_hidden.T))
                self.OUTPUT_L2 = self.activation((np.dot(self.OUTPUT_L1, self.WEIGHT_output) + self.BIAS_output.T))
                'Stage 2 - One-Hot-Encoding'
                if(y[idx] == 0): 
                    self.output = np.array([1,0,0]) #Class1 {1,0,0}
                elif(y[idx] == 1):
                    self.output = np.array([0,1,0]) #Class2 {0,1,0}
                elif(y[idx] == 2):
                    self.output = np.array([0,0,1]) #Class3 {0,0,1}
                
                square_error = 0
                for i in range(self.OutputLayer):
                    erro = (self.output[i] - self.OUTPUT_L2[i])**2
                    square_error = (square_error + (0.05 * erro))
                    total_error = total_error + square_error
         
                'Backpropagation : Update Weights'
                self.Backpropagation_Algorithm(inputs)
                
            total_error = (total_error / n)
            if((count_epoch % 50 == 0)or(count_epoch == 1)):
                print("Epoch ", count_epoch, "- Total Error: ",total_error)
                error_array.append(total_error)
                epoch_array.append(count_epoch)
                
            W0.append(self.WEIGHT_hidden)
            W1.append(self.WEIGHT_output)
             
                
            count_epoch += 1
        self.show_err_graphic(error_array,epoch_array)
        
        plt.plot(W0[0])
        plt.title('Weight Hidden update during training')
        plt.legend(['neuron1', 'neuron2', 'neuron3', 'neuron4', 'neuron5'])
        plt.ylabel('Value Weight')
        plt.show()
        
        plt.plot(W1[0])
        plt.title('Weight Output update during training')
        plt.legend(['neuron1', 'neuron2', 'neuron3'])
        plt.ylabel('Value Weight')
        plt.show()

        return self

## Finding the best parameters 

<p style="text-align: justify;">For find the best parameters, it was necessary to realize various tests using different values to the parameters. The graphs below denote all tests made to select the best configuration for the multilayer perceptron. These tests were important in selecting the best settings and ensuring the best accuracy. The graph was drawn manually, but you can change the settings and note the results obtained. The tests involve different activation functions and the number of neurons for each layer.</p>

In [ ]:
def show_test():
    ep1 = [0,100,200,300,400,500,600,700,800,900,1000,1500,2000]
    h_5 = [0,60,70,70,83.3,93.3,96.7,86.7,86.7,76.7,73.3,66.7,66.7]
    h_4 = [0,40,70,63.3,66.7,70,70,70,70,66.7,66.7,43.3,33.3]
    h_3 = [0,46.7,76.7,80,76.7,76.7,76.6,73.3,73.3,73.3,73.3,76.7,76.7]
    plt.figure(figsize=(10,4))
    l1, = plt.plot(ep1, h_3, "--",color='b',label="node-3", marker=11)
    l2, = plt.plot(ep1, h_4, "--",color='g',label="node-4", marker=8)
    l3, = plt.plot(ep1, h_5, "--",color='r',label="node-5", marker=5)
    plt.legend(handles=[l1,l2,l3], loc=1)
    plt.xlabel("number of Epochs")
    plt.ylabel("% Hits")
    plt.title("Number of Hidden Layers - Performance")
    
    ep2 = [0,100,200,300,400,500,600,700]
    tanh = [0.18,0.027,0.025,0.022,0.0068,0.0060,0.0057,0.00561]
    sigm = [0.185,0.0897,0.060,0.0396,0.0343,0.0314,0.0296,0.0281]
    Relu = [0.185,0.05141,0.05130,0.05127,0.05124,0.05123,0.05122,0.05121]
    plt.figure(figsize=(10,4))
    l1 , = plt.plot(ep2, tanh, "--",color='b',label="Hyperbolic Tangent",marker=11)
    l2 , = plt.plot(ep2, sigm, "--",color='g',label="Sigmoide", marker=8)
    l3 , = plt.plot(ep2, Relu, "--",color='r',label="ReLu", marker=5)
    plt.legend(handles=[l1,l2,l3], loc=1)
    plt.xlabel("Epoch")
    plt.ylabel("Error")
    plt.title("Activation Functions - Performance")
    
    fig, ax = plt.subplots()
    names = ["Hyperbolic Tangent","Sigmoide","ReLU"]
    x1 = [2.0,4.0,6.0]
    plt.bar(x1[0], 53.4,0.4,color='b')
    plt.bar(x1[1], 96.7,0.4,color='g')
    plt.bar(x1[2], 33.2,0.4,color='r')
    plt.xticks(x1,names)
    plt.ylabel('% Hits')
    plt.title('Hits - Activation Functions')
    plt.show()

In [ ]:
show_test()

# Training the Artificial Neural Network(MLP)

## Step 1: training our MultiLayer Perceptron

In [ ]:
dictionary = {'InputLayer':4, 'HiddenLayer':5, 'OutputLayer':3,
              'Epocas':700, 'LearningRate':0.005,'BiasHiddenValue':-1, 
              'BiasOutputValue':-1, 'ActivationFunction':'sigmoid'}

Perceptron = MultiLayerPerceptron(dictionary)
Perceptron.fit(train_X,train_y)

## Step 2: testing our results 

In [ ]:
prev, dataframe = Perceptron.predict(test_X, test_y)
hits = n_set = n_vers = n_virg = 0
score_set = score_vers = score_virg = 0
for j in range(len(test_y)):
    if(test_y[j] == 0): n_set += 1
    elif(test_y[j] == 1): n_vers += 1
    elif(test_y[j] == 2): n_virg += 1
        
for i in range(len(test_y)):
    if test_y[i] == prev[i]: 
        hits += 1
    if test_y[i] == prev[i] and test_y[i] == 0:
        score_set += 1
    elif test_y[i] == prev[i] and test_y[i] == 1:
        score_vers += 1
    elif test_y[i] == prev[i] and test_y[i] == 2:
        score_virg += 1    
         
hits = (hits / len(test_y)) * 100
faults = 100 - hits

In [ ]:
dataframe

## Step 3. Accuracy and precision the Multilayer Perceptron

In [ ]:
graph_hits = []
print("Porcents :","%.2f"%(hits),"% hits","and","%.2f"%(faults),"% faults")
print("Total samples of test",n_samples)
print("*Iris-Setosa:",n_set,"samples")
print("*Iris-Versicolour:",n_vers,"samples")
print("*Iris-Virginica:",n_virg,"samples")

graph_hits.append(hits)
graph_hits.append(faults)
labels = 'Hits', 'Faults';
sizes = [96.5, 3.3]
explode = (0, 0.14)

fig1, ax1 = plt.subplots();
ax1.pie(graph_hits, explode=explode,colors=['green','red'],labels=labels, autopct='%1.1f%%',
shadow=True, startangle=90)
ax1.axis('equal')
plt.show()

## Step 4. Score for each one of the samples

In [ ]:
acc_set = (score_set/n_set)*100
acc_vers = (score_vers/n_vers)*100
acc_virg = (score_virg/n_virg)*100
print("- Acurracy Iris-Setosa:","%.2f"%acc_set, "%")
print("- Acurracy Iris-Versicolour:","%.2f"%acc_vers, "%")
print("- Acurracy Iris-Virginica:","%.2f"%acc_virg, "%")
names = ["Setosa","Versicolour","Virginica"]
x1 = [2.0,4.0,6.0]
fig, ax = plt.subplots()
r1 = plt.bar(x1[0], acc_set,color='orange',label='Iris-Setosa')
r2 = plt.bar(x1[1], acc_vers,color='green',label='Iris-Versicolour')
r3 = plt.bar(x1[2], acc_virg,color='purple',label='Iris-Virginica')
plt.ylabel('Scores %')
plt.xticks(x1, names);plt.title('Scores by iris flowers - Multilayer Perceptron')
plt.show()